### safe Parallel Code

In [3]:
import numpy as np
import threading
import time
from PIL import Image

In [4]:
image_path = "test_images/large_1024x1024.png"
image = np.array(Image.open(image_path).convert("L"), dtype=np.float64)

In [5]:
gaussian_kernel = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=np.float64) / 16

In [6]:
# Race condition 1
def pixel_worker(shared_output, partial_sum, repeats):

    for _ in range(repeats):

        temp = shared_output[0]
        time.sleep(0)            # for thread switching
        shared_output[0] = temp + partial_sum


def test_output_pixel(n_workers=3):

    # center pixel
    i, j = image.shape[0] // 2, image.shape[1] // 2

    partial_sums = []

    for k in range(n_workers):

        partial = 0

        for l in range(3):
            partial += (
                image[i + k - 1, j + l - 1]
                * gaussian_kernel[k, l]
            )

        partial_sums.append(partial)

    shared_output = [0.0]
    repeats = 1000

    threads = [
        threading.Thread(
            target=pixel_worker,
            args=(shared_output, partial, repeats)
        )
        for partial in partial_sums
    ]

    start = time.perf_counter()

    for t in threads:
        t.start()

    for t in threads:
        t.join()

    elapsed = time.perf_counter() - start

    expected = sum(partial_sums) * repeats

    print("Expected:", expected)
    print("Actual:", shared_output[0])
    print("Absolute error:", abs(expected - shared_output[0]))
    print(f"Execution time: {elapsed:.4f}")

In [7]:
for run in range(5):

    print(f"\n RUN {run + 1}")
    test_output_pixel()


 RUN 1
Expected: 18062.5
Actual: 5161.5
Absolute error: 12901.0
Execution time: 0.0969

 RUN 2
Expected: 18062.5
Actual: 3812.5
Absolute error: 14250.0
Execution time: 0.0720

 RUN 3
Expected: 18062.5
Actual: 5132.625
Absolute error: 12929.875
Execution time: 0.0785

 RUN 4
Expected: 18062.5
Actual: 9128.8125
Absolute error: 8933.6875
Execution time: 0.0758

 RUN 5
Expected: 18062.5
Actual: 5128.8125
Absolute error: 12933.6875
Execution time: 0.0764


Expected: 18062.5
Actual: 9125.0
Absolute error: 8937.5
Execution time: 0.1659

 RUN 3
Expected: 18062.5
Actual: 5150.875
Absolute error: 12911.625
Execution time: 0.1701

 RUN 4
Expected: 18062.5
Actual: 9128.8125
Absolute error: 8933.6875
Execution time: 0.1563

 RUN 5
Expected: 18062.5
Actual: 9128.8125
Absolute error: 8933.6875
Execution time: 0.1101


In [8]:
# Safe Race Condition 1: Shared Pixel using Lock

def safe_pixel_worker(shared_output, partial_sum, repeats, lock):

    for _ in range(repeats):

        with lock:  # Critical Section
            temp = shared_output[0]
            shared_output[0] = temp + partial_sum


def test_safe_output_pixel(n_workers=3):

    # Select the center pixel
    i, j = image.shape[0] // 2, image.shape[1] // 2

    partial_sums = []

    # Calculate partial sums
    for k in range(n_workers):

        partial = 0

        for l in range(3):
            partial += (
                image[i + k - 1, j + l - 1]
                * gaussian_kernel[k, l]
            )

        partial_sums.append(partial)

    shared_output = [0.0]
    repeats = 1000

    # Create a lock to protect shared output
    lock = threading.Lock()

    threads = [
        threading.Thread(
            target=safe_pixel_worker,
            args=(shared_output, partial, repeats, lock)
        )
        for partial in partial_sums
    ]

    start = time.perf_counter()

    for t in threads:
        t.start()

    for t in threads:
        t.join()

    elapsed = time.perf_counter() - start

    expected = sum(partial_sums) * repeats

    print("Expected:", expected)
    print("Actual:", shared_output[0])
    print("Absolute error:", abs(expected - shared_output[0]))
    print(f"Execution time: {elapsed:.6f} seconds")


# Run the safe experiment
for run in range(5):

    print(f"\nSAFE PIXEL RUN {run + 1}")
    test_safe_output_pixel()


SAFE PIXEL RUN 1
Expected: 18062.5
Actual: 18062.5
Absolute error: 0.0
Execution time: 0.003515 seconds

SAFE PIXEL RUN 2
Expected: 18062.5
Actual: 18062.5
Absolute error: 0.0
Execution time: 0.004374 seconds

SAFE PIXEL RUN 3
Expected: 18062.5
Actual: 18062.5
Absolute error: 0.0
Execution time: 0.002606 seconds

SAFE PIXEL RUN 4
Expected: 18062.5
Actual: 18062.5
Absolute error: 0.0
Execution time: 0.002586 seconds

SAFE PIXEL RUN 5
Expected: 18062.5
Actual: 18062.5
Absolute error: 0.0
Execution time: 0.002637 seconds


In [9]:
# Race condition 2
def counter_worker(counter, iterations):

    for _ in range(iterations):

        temp = counter[0]
        time.sleep(0)  
        counter[0] = temp + 1


def test_counter(n_workers=2):

    counter = [0]
    iterations = 50000

    threads = [
        threading.Thread(
            target=counter_worker,
            args=(counter, iterations)
        )
        for _ in range(n_workers)
    ]

    start = time.perf_counter()

    for t in threads:
        t.start()

    for t in threads:
        t.join()

    elapsed = time.perf_counter() - start

    expected = n_workers * iterations

    print("Expected:", expected)
    print("Actual:", counter[0])
    print("Lost updates:", expected - counter[0])
    print(f"Execution time: {elapsed:.4f}")

In [10]:
for run in range(5):

    print(f"\n RUN {run + 1}")
    test_counter()


 RUN 1
Expected: 100000
Actual: 50000
Lost updates: 50000
Execution time: 3.6555

 RUN 2
Expected: 100000
Actual: 50001
Lost updates: 49999
Execution time: 3.7513

 RUN 3
Expected: 100000
Actual: 50000
Lost updates: 50000
Execution time: 3.8277

 RUN 4
Expected: 100000
Actual: 50001
Lost updates: 49999
Execution time: 3.6386

 RUN 5
Expected: 100000
Actual: 50000
Lost updates: 50000
Execution time: 3.6551


In [11]:
# Safe Race Condition 2: Shared Counter using Lock

def safe_counter_worker(counter, iterations, lock):

    for _ in range(iterations):

        with lock:  # Critical Section
            counter[0] += 1


def test_safe_counter(n_workers=2):

    counter = [0]
    iterations = 50000

    # Create a lock to protect the shared counter
    lock = threading.Lock()

    threads = [
        threading.Thread(
            target=safe_counter_worker,
            args=(counter, iterations, lock)
        )
        for _ in range(n_workers)
    ]

    start = time.perf_counter()

    for t in threads:
        t.start()

    for t in threads:
        t.join()

    elapsed = time.perf_counter() - start

    expected = n_workers * iterations

    print("Expected:", expected)
    print("Actual:", counter[0])
    print("Lost updates:", expected - counter[0])
    print(f"Execution time: {elapsed:.6f} seconds")


# Run the safe experiment
for run in range(5):

    print(f"\nSAFE COUNTER RUN {run + 1}")
    test_safe_counter()


SAFE COUNTER RUN 1
Expected: 100000
Actual: 100000
Lost updates: 0
Execution time: 0.047656 seconds

SAFE COUNTER RUN 2
Expected: 100000
Actual: 100000
Lost updates: 0
Execution time: 0.039372 seconds

SAFE COUNTER RUN 3
Expected: 100000
Actual: 100000
Lost updates: 0
Execution time: 0.037836 seconds

SAFE COUNTER RUN 4
Expected: 100000
Actual: 100000
Lost updates: 0
Execution time: 0.037194 seconds

SAFE COUNTER RUN 5
Expected: 100000
Actual: 100000
Lost updates: 0
Execution time: 0.065005 seconds
